# Pydantic

### Pydantic basics for data validation - Repetition

- fields and methods
- pydantic enables work with data in OOP manner
- validate data
- output json data (from instance -> json)

Pydantic model created by inheriting from BaseModel (subclass, children - inheritance fields)

In [1]:
import requests

data = requests.get("https://pokeapi.co/api/v2/pokemon?limit=20").json()
data.keys()

dict_keys(['count', 'next', 'previous', 'results'])

# Start without pydantic

In [16]:
#defined a class Person (blueprint)
class Person:       #dunder init in python (or create instance from class)
    def __init__(self, name, gender, age):
        self.name = name
        self.gender = gender
        self.age = age

# instantiated an instance from the Person class       
person1 = Person(name= "Susanna", age="45", gender= "F") #default dunder init (wrapper)
person1 # person1 is injected into 'self' and it has the attributes


In [17]:
# attribute in this person1 instance
person1.name, #attribute with a value
person1.age,
person1.gender

'F'

In [18]:
# Python is permissive 'no validation'
person2 = Person(name= 312+8, gender=True, age="minus 10")
person2.name, person2.gender, person2.age

(320, True, 'minus 10')

What if we type hint?

In [19]:
# used type hinting to 'hint' at dtypes for attributes
class Person:
    def __init__(self, name: str, gender: str, age: int):
        self.name = name
        self.gender = gender
        self.age = age

person3 = Person("Gauss", "M", 70)
person3

In [20]:
# despite type hinting no errors...
person4 = Person(3.1416, 2.714, "fisk")
person4.age

'fisk'

### Validate the Person class

In [23]:
class Person:
    def __init__(self, name: str, gender: str, age: int):
        if not isinstance(name, str):
            raise TypeError(f"name must be of type str not {type(name)}")
        self.name = name

        self.gender = gender
        self.age = age
try:
    Person(name= 532, gender = 3, age= "hare")
except TypeError as err:
    print(err)

name must be of type str not <class 'int'>


In [30]:
# validation in normal python class
class Person:
    def __init__(self, name: str, gender: str, age: int):
        if not isinstance(name, str):
            raise TypeError(f"name must be of type str not {type(name)}")
        self.name = name

        self.gender = gender
        self.age = age

    # getter ( see also 'setter'. compare C# 'get' 'set')
    @property
    def age(self):
        return self._age #underscore for 'private' varible
    
    @age.setter
    def age(self, value: int):
        if not isinstance(value, int):
            raise TypeError(f"age must be of type int not {type(age)} that you have provided")
        if value < 0 or value > 125:
            raise ValueError(f"age must between 0 and 125, not {value} that you provided")
        
        self._age = value

person5 = Person(name= "Bella", gender="F", age= 3)
person5.age
    

3

## work smarter with pydantic

In [37]:
from pydantic import BaseModel

# inherits from Basemodel -> makes it into pydantic model, but still a normal python class
# declare a class using BaseModel
class Person(BaseModel):
    name: str
    gender: str
    age: int

person6 = Person(name="Agge", age=32, gender="M")
person6


Person(name='Agge', gender='M', age=32)

In [34]:
person6.name, person6.age, person6.gender

('Agge', 32, 'M')

In [35]:
person6.age= 40
person6.age

40

In [36]:
person6

Person(name='Agge', gender='M', age=40)

In [ ]:
Person(name= 3.1415, age= "hare", gender= "trehundra") # this will crash

In [41]:
from pydantic import ValidationError
try:
    Person(name= 532, gender = 3, age= "hare")
except ValidationError as err:
    print(err)

3 validation errors for Person
name
  Input should be a valid string [type=string_type, input_value=532, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type
gender
  Input should be a valid string [type=string_type, input_value=3, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='hare', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/int_parsing


In [42]:
try:
    Person(name= "Sofia", gender = 3.1415, age= -5)
except ValidationError as err:
    print(err)

1 validation error for Person
gender
  Input should be a valid string [type=string_type, input_value=3.1415, input_type=float]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type


In [44]:
# negative age validates w/o error
Person(name= "Sofia", gender = "F", age= -5)

Person(name='Sofia', gender='F', age=-5)

In [ ]:
#fix person class to validate age range (non-negative, and within range)
from pydantic import Field

class Person(BaseModel):
    name: str
    gender: str
    age: int = Field(gt= -1, lt = 126)

person7 = Person(name="Minna", gender="F", age=-5) #this will thrown an error

In [46]:
class Person(BaseModel):
    name: str
    gender: str
    age: int = Field(gt= -1, lt = 126)
try:
    Person(name= "Sofia", gender = 3.1415, age= 126)
except ValidationError as err:
    print(err)

2 validation errors for Person
gender
  Input should be a valid string [type=string_type, input_value=3.1415, input_type=float]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type
age
  Input should be less than 126 [type=less_than, input_value=126, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/less_than


In [56]:
from typing import Literal

class Person(BaseModel):
    name: str
    gender: str = Literal["M", "F", "Other"]
    age: int = Field(gt= -1, lt = 126)

try:
    Person(name= "Sofia", gender = "Female", age= 24)
except ValidationError as err:
    print(err)


Person(name='Sofia', gender='F', age=24)

In [57]:
person7 = Person(name= "Sofia", gender = "F", age= 24)
person7

Person(name='Sofia', gender='F', age=24)

make an instance into a dictionary

In [59]:
person7.model_dump() #gives dict

{'name': 'Sofia', 'gender': 'F', 'age': 24}

Serialization - make instance into a json string

In [61]:
# serialization
person7.model_dump_json() #gives json str formats of key-value pairs

'{"name":"Sofia","gender":"F","age":24}'